# A-share 5-minute Stage 3: MLP + GRU
Runs 8 rolling windows × 5 seeds with delayed-entry evaluation. The runner auto-discovers the uploaded dataset root and writes all reports to `/kaggle/working/stage3_report`.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile
import torch

matches = list(Path('/kaggle/input').rglob('run_neural_models.py'))
assert len(matches) == 1, f'Expected one runner, found: {matches}'
input_project = matches[0].parent
assert torch.cuda.is_available(), 'Enable a GPU accelerator before running'
project = Path('/kaggle/working/stage3_project')
project.mkdir(parents=True, exist_ok=True)
for name in ['config.py', 'build_5min_dataset.py', 'run_baseline.py', 'run_neural_models.py']:
    shutil.copy2(input_project / name, project / name)
(project / 'data').mkdir(exist_ok=True)
for name in ['features_5min', 'dataset']:
    target = project / 'data' / name
    if not target.exists():
        target.symlink_to(input_project / 'data' / name, target_is_directory=True)
baseline_matches = list(Path('/kaggle/input').rglob('baseline_report/window_metrics.csv'))
assert len(baseline_matches) == 1, f'Expected one baseline report, found: {baseline_matches}'
shutil.copytree(baseline_matches[0].parent, project / 'data/baseline_report', dirs_exist_ok=True)
output = Path('/kaggle/working/stage3_report')
output.mkdir(parents=True, exist_ok=True)
manifest = {
    'input_project': str(input_project),
    'working_project': str(project),
    'torch': torch.__version__,
    'cuda_device': torch.cuda.get_device_name(0),
}
print(json.dumps(manifest, ensure_ascii=False, indent=2))
cmd = [
    sys.executable, '-u', str(project / 'run_neural_models.py'),
    '--models', 'MLP', 'GRU',
    '--windows', '1', '2', '3', '4', '5', '6', '7', '8',
    '--seeds', '20260807', '20260808', '20260809', '20260810', '20260811',
    '--device', 'cuda',
    '--flat-dir', str(project / 'data/dataset/flat'),
    '--feature-dir', str(project / 'data/features_5min'),
    '--seq-dir', str(project / 'data/dataset/seq'),
    '--output-dir', str(output),
]
started = time.time()
subprocess.run(cmd, check=True, cwd=project)
manifest['elapsed_hours'] = (time.time() - started) / 3600
(output / 'kaggle_runtime.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
archive = Path('/kaggle/working/stage3_report.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in output.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(output.parent))
print(f'Completed in {manifest["elapsed_hours"]:.2f} h: {archive}')